In [ ]:
from pathlib import Path

for split in ["train", "test", "val"]:
    for label in ["real", "fake"]:
        count = len(list(Path(f"../data/processed/{split}/{label}").glob("*.jpg")))
        print(f"{split} {label} {count}")



In [ ]:
import torch
from torch.utils.data import Dataset
import torchvision
import cv2
from torchvision import transforms

In [ ]:
class DeepfakeDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.images = list(Path(root_dir).rglob("*.jpg"))
        self.labels = [0 if image.parent.name=="real" else 1 for image in self.images]
        self.transform=transform
    def __len__(self):
        return len(self.images)
    def __getitem__(self,idx):
        image = cv2.imread(str(self.images[idx]))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        if self.transform is not None:
            image = self.transform(image)
        return image, self.labels[idx]
        

In [ ]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std = [0.229,0.224,0.225])
])

train_dataset = DeepfakeDataset(Path("../data/processed/train/"), transform=transform)

In [ ]:
print(f"Total samples: {len(train_dataset)}")
print(f"{train_dataset[0]}")
print(f"{type(train_dataset[1])}")
image, label = train_dataset[0]
print(f"label: {label} ")
print(f"{image.min()} to {image.max()}")
print(f"{image.shape}")

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4)
print(f"{len(train_loader)}")

In [ ]:
images, labels = next(iter(train_loader))
print(f"{images.shape}")
print(f"{labels.shape}")


In [ ]:
test_dataset = DeepfakeDataset(Path("../data/processed/test/"), transform=transform)
val_dataset = DeepfakeDataset(Path("../data/processed/val/"), transform=transform)

test_loader= DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4)

In [ ]:
print(f"val: {len(val_dataset)}")
print(f"test: {len(test_dataset)}")
print(f"train: {len(train_dataset)}")